In [ ]:
%pip install -q torch-geometric pyarrow


In [ ]:
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Evaluation device:", device)


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
PROJECT_DIR = Path("/content/drive/MyDrive/BD_LAB2")
DATA_DIR = PROJECT_DIR / "db"
CHECKPOINT_PATH = PROJECT_DIR / "training_outputs" / "gcn_checkpoint.pt"
EVALUATOR_PATH = PROJECT_DIR / "scripts" / "evaluate.py"
OUTPUT_DIR = PROJECT_DIR / "evaluation_outputs"

cutoff_date = pd.Timestamp("2026-11-01")
week_end = cutoff_date + pd.Timedelta(days=7)
threshold = 0.7


In [ ]:
for required_file in [CHECKPOINT_PATH, EVALUATOR_PATH]:
    if not required_file.is_file():
        raise FileNotFoundError(f"Place the required file here: {required_file}")
if not 0 < threshold < 1:
    raise ValueError("The probability threshold must be between 0 and 1.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

checkpoint = torch.load(
    CHECKPOINT_PATH, map_location="cpu", weights_only=True
)
if pd.Timestamp(checkpoint["training_week_end"]) > cutoff_date:
    raise ValueError("The validation week overlaps the training target week.")

activity_columns = [
    "open_pr", "open_issue", "issue_comment",
    "pr_review", "pr_review_comment", "push",
]
node_type_order = [
    "users", "repositories", "pull_requests", "issues",
    "pr_reviews", "issue_comments", "pr_review_comments",
]
assert checkpoint["activity_columns"] == activity_columns
assert checkpoint["node_type_order"] == node_type_order

print("Training target week ended:", checkpoint["training_week_end"])
print("Validation week:", cutoff_date, "to", week_end)
print("Completed training epochs:", checkpoint["epochs_completed"])


In [ ]:
import pandas as pd

issues_comments = pd.read_parquet(DATA_DIR / "IssueComments.parquet")
issues = pd.read_parquet(DATA_DIR / "Issues.parquet")
pr_review_comments = pd.read_parquet(DATA_DIR / "PRReviewComments.parquet")
pr_reviews = pd.read_parquet(DATA_DIR / "PRReviews.parquet")
pull_requests = pd.read_parquet(DATA_DIR / "PullRequests.parquet")
pushes = pd.read_parquet(DATA_DIR / "Pushes.parquet")
reposetories = pd.read_parquet(DATA_DIR / "Repositories.parquet")
users = pd.read_parquet(DATA_DIR / "Users.parquet")

num_users = len(users)
num_repos = len(reposetories)
num_issues = len(issues)
num_prs = len(pull_requests)
num_pushes = len(pushes)
num_pr_reviews = len(pr_reviews)
num_pr_review_comments = len(pr_review_comments)
num_issue_comments = len(issues_comments)

print("issue_comments shape:", issues_comments.shape)
print("issues shape:", issues.shape)
print("pr_review_comments shape:", pr_review_comments.shape)
print("pr_reviews shape:", pr_reviews.shape)
print("pull_requests shape:", pull_requests.shape)
print("pushes shape:", pushes.shape)
print("reposetories shape:", reposetories.shape)


In [ ]:
# Normalize table times to the UTC convention used by the assignment.
for table in [
    users, reposetories, pull_requests, issues,
    pr_reviews, issues_comments, pr_review_comments, pushes,
]:
    for column in ["first_seen_at", "created_at", "submitted_at"]:
        if column in table:
            table[column] = pd.to_datetime(
                table[column], utc=True, errors="raise"
            ).dt.tz_localize(None)

# The existing offset scheme requires consecutive primary keys.
for table, primary_key in [
    (users, "user_id"), (reposetories, "repo_id"),
    (pull_requests, "pr_id"), (issues, "issue_id"),
    (pr_reviews, "review_id"), (issues_comments, "comment_id"),
    (pr_review_comments, "comment_id"),
]:
    assert np.array_equal(table[primary_key].to_numpy(), np.arange(len(table)))

coverage = pd.DataFrame([
    {
        "table": name,
        "first_observed": table[time_column].min(),
        "last_observed": table[time_column].max(),
    }
    for name, table, time_column in [
        ("PullRequests", pull_requests, "first_seen_at"),
        ("Issues", issues, "first_seen_at"),
        ("IssueComments", issues_comments, "created_at"),
        ("PRReviews", pr_reviews, "first_seen_at"),
        ("PRReviewComments", pr_review_comments, "created_at"),
        ("Pushes", pushes, "created_at"),
    ]
])
display(coverage)

# This detects obviously short data coverage. Inspect the downloaded
# hourly archive files separately to establish complete coverage.
latest_push = pushes["created_at"].max()
if pd.isna(latest_push) or latest_push < week_end - pd.Timedelta(hours=1):
    raise ValueError("The loaded push data ends before the validation week is covered.")


In [ ]:
from torch_geometric.nn import GCNConv

class GCN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, output_dim)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, training=self.training)
        x = self.conv2(x, edge_index)
        return x


In [ ]:
model = GCN(**checkpoint["model_config"]).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(model)


In [ ]:
week_end = cutoff_date + pd.Timedelta(days=7)

past_pushes = pushes[pushes["created_at"] < cutoff_date]

week_pushes = pushes[  # Filter pushes that occurred within the week after the cutoff date
    (pushes["created_at"] >= cutoff_date) &
    (pushes["created_at"] < week_end)
]

past_pull_requests = pull_requests[pull_requests["first_seen_at"] < cutoff_date]

week_pull_requests = pull_requests[  # Filter pull requests that occurred within the week after the cutoff date
    (pull_requests["created_at"] >= cutoff_date) &
    (pull_requests["created_at"] < week_end)
]

past_pr_reviews = pr_reviews[pr_reviews["first_seen_at"] < cutoff_date]

week_pr_reviews = pr_reviews[  # Filter pull request reviews that occurred within the week after the cutoff date
    (pr_reviews["submitted_at"] >= cutoff_date) &
    (pr_reviews["submitted_at"] < week_end)
]

past_issues = issues[issues["first_seen_at"] < cutoff_date]

week_issues = issues[
    (issues["created_at"] >= cutoff_date) &
    (issues["created_at"] < week_end) &
    (~issues["is_pull_request"])
]

past_issue_comments = issues_comments[issues_comments["created_at"] < cutoff_date]

week_issue_comments = issues_comments[  # Filter issue comments that occurred within the week after the cutoff date
    (issues_comments["created_at"] >= cutoff_date) &
    (issues_comments["created_at"] < week_end)
]

past_pr_review_comments = pr_review_comments[pr_review_comments["created_at"] < cutoff_date]

week_pr_review_comments = pr_review_comments[  # Filter pull request review comments that occurred within the week after the cutoff date
    (pr_review_comments["created_at"] >= cutoff_date) &
    (pr_review_comments["created_at"] < week_end)
]

# Train on cohort users already observed at the prediction cutoff.
targets = users.loc[
    users["is_cohort"] & (users["first_seen_at"] < cutoff_date),
    ["user_id"],
].copy()


In [ ]:
user_nodes = torch.tensor(
    past_pushes["user_id"].to_numpy(), dtype=torch.long
)
repo_nodes = torch.tensor(
    past_pushes["repo_id"].to_numpy() + num_users, dtype=torch.long
)

push_edges = torch.stack([user_nodes, repo_nodes], dim=0)

print(push_edges.shape)
print(push_edges[:, :5])

reverse_push_edges = push_edges.flip(0)

edge_index = torch.cat([push_edges, reverse_push_edges], dim=1)

print(edge_index.shape)
print(edge_index[:, :5])


In [ ]:
pr_links = past_pull_requests[["pr_id", "author_id", "repo_id"]].dropna()

pr_nodes = torch.tensor(
    pr_links["pr_id"].to_numpy(dtype=np.int64) + num_users + num_repos,
    dtype=torch.long,
)

pr_repo_nodes = torch.tensor(
    pr_links["repo_id"].to_numpy(dtype=np.int64) + num_users,
    dtype=torch.long,
)

pr_author_nodes = torch.tensor(
    pr_links["author_id"].to_numpy(dtype=np.int64),
    dtype=torch.long,
)

pr_to_author = torch.stack([pr_nodes, pr_author_nodes], dim=0)
pr_to_repo = torch.stack([pr_nodes, pr_repo_nodes], dim=0)

pr_edges = torch.cat([pr_to_author, pr_to_repo], dim=1)
reverse_pr_edges = pr_edges.flip(0)
pr_edges = torch.cat([pr_edges, reverse_pr_edges], dim=1)

edge_index = torch.cat([edge_index, pr_edges], dim=1)


In [ ]:
issue_offset = num_users + num_repos + num_prs

issue_author_links = past_issues[["issue_id", "author_id"]].dropna()
issue_to_author = torch.stack([
    torch.tensor(
        issue_author_links["issue_id"].to_numpy(dtype=np.int64) + issue_offset,
        dtype=torch.long,
    ),
    torch.tensor(
        issue_author_links["author_id"].to_numpy(dtype=np.int64),
        dtype=torch.long,
    ),
], dim=0)

issue_repo_links = past_issues[["issue_id", "repo_id"]].dropna()
issue_to_repo = torch.stack([
    torch.tensor(
        issue_repo_links["issue_id"].to_numpy(dtype=np.int64) + issue_offset,
        dtype=torch.long,
    ),
    torch.tensor(
        issue_repo_links["repo_id"].to_numpy(dtype=np.int64) + num_users,
        dtype=torch.long,
    ),
], dim=0)

issue_assignee_links = past_issues[["issue_id", "assignee_id"]].dropna()
issue_to_assignee = torch.stack([
    torch.tensor(
        issue_assignee_links["issue_id"].to_numpy(dtype=np.int64) + issue_offset,
        dtype=torch.long,
    ),
    torch.tensor(
        issue_assignee_links["assignee_id"].to_numpy(dtype=np.int64),
        dtype=torch.long,
    ),
], dim=0)

issue_pr_links = past_issues[["issue_id", "pull_request_id"]].dropna()
issue_to_pr = torch.stack([
    torch.tensor(
        issue_pr_links["issue_id"].to_numpy(dtype=np.int64) + issue_offset,
        dtype=torch.long,
    ),
    torch.tensor(
        issue_pr_links["pull_request_id"].to_numpy(dtype=np.int64)
        + num_users + num_repos,
        dtype=torch.long,
    ),
], dim=0)

issue_edges = torch.cat([
    issue_to_author, issue_to_repo, issue_to_assignee, issue_to_pr,
], dim=1)
reverse_issue_edges = issue_edges.flip(0)
issue_edges = torch.cat([issue_edges, reverse_issue_edges], dim=1)

edge_index = torch.cat([edge_index, issue_edges], dim=1)

print("Issue edges including reverse connections:", issue_edges.shape)


In [ ]:
# PR review nodes follow the issue nodes.
review_offset = issue_offset + num_issues

review_to_reviewer_links = past_pr_reviews[["review_id", "reviewer_id"]].dropna()
review_to_reviewer = torch.stack([
    torch.tensor(
        review_to_reviewer_links["review_id"].to_numpy(dtype=np.int64) + review_offset,
        dtype=torch.long,
    ),
    torch.tensor(
        review_to_reviewer_links["reviewer_id"].to_numpy(dtype=np.int64),
        dtype=torch.long,
    ),
], dim=0)

review_to_repo_links = past_pr_reviews[["review_id", "repo_id"]].dropna()
review_to_repo = torch.stack([
    torch.tensor(
        review_to_repo_links["review_id"].to_numpy(dtype=np.int64) + review_offset,
        dtype=torch.long,
    ),
    torch.tensor(
        review_to_repo_links["repo_id"].to_numpy(dtype=np.int64) + num_users,
        dtype=torch.long,
    ),
], dim=0)

review_to_pr_links = past_pr_reviews[["review_id", "pr_id"]].dropna()
review_to_pr = torch.stack([
    torch.tensor(
        review_to_pr_links["review_id"].to_numpy(dtype=np.int64) + review_offset,
        dtype=torch.long,
    ),
    torch.tensor(
        review_to_pr_links["pr_id"].to_numpy(dtype=np.int64) + num_users + num_repos,
        dtype=torch.long,
    ),
], dim=0)

review_edges = torch.cat([
    review_to_reviewer, review_to_repo, review_to_pr,
], dim=1)
reverse_review_edges = review_edges.flip(0)
review_edges = torch.cat([review_edges, reverse_review_edges], dim=1)

edge_index = torch.cat([edge_index, review_edges], dim=1)

print("Edges including reverse connections:", review_edges.shape)


In [ ]:
# Issue comment nodes follow the PR review nodes.
issue_comment_offset = review_offset + num_pr_reviews

issue_comment_to_author_links = past_issue_comments[["comment_id", "author_id"]].dropna()
issue_comment_to_author = torch.stack([
    torch.tensor(
        issue_comment_to_author_links["comment_id"].to_numpy(dtype=np.int64) + issue_comment_offset,
        dtype=torch.long,
    ),
    torch.tensor(
        issue_comment_to_author_links["author_id"].to_numpy(dtype=np.int64),
        dtype=torch.long,
    ),
], dim=0)

issue_comment_to_repo_links = past_issue_comments[["comment_id", "repo_id"]].dropna()
issue_comment_to_repo = torch.stack([
    torch.tensor(
        issue_comment_to_repo_links["comment_id"].to_numpy(dtype=np.int64) + issue_comment_offset,
        dtype=torch.long,
    ),
    torch.tensor(
        issue_comment_to_repo_links["repo_id"].to_numpy(dtype=np.int64) + num_users,
        dtype=torch.long,
    ),
], dim=0)

issue_comment_to_issue_links = past_issue_comments[["comment_id", "issue_id"]].dropna()
issue_comment_to_issue = torch.stack([
    torch.tensor(
        issue_comment_to_issue_links["comment_id"].to_numpy(dtype=np.int64) + issue_comment_offset,
        dtype=torch.long,
    ),
    torch.tensor(
        issue_comment_to_issue_links["issue_id"].to_numpy(dtype=np.int64) + issue_offset,
        dtype=torch.long,
    ),
], dim=0)

issue_comment_edges = torch.cat([
    issue_comment_to_author, issue_comment_to_repo, issue_comment_to_issue,
], dim=1)
reverse_issue_comment_edges = issue_comment_edges.flip(0)
issue_comment_edges = torch.cat([issue_comment_edges, reverse_issue_comment_edges], dim=1)

edge_index = torch.cat([edge_index, issue_comment_edges], dim=1)

print("Edges including reverse connections:", issue_comment_edges.shape)


In [ ]:
# PR review comment nodes follow the issue comment nodes.
pr_review_comment_offset = issue_comment_offset + num_issue_comments

pr_review_comment_to_author_links = past_pr_review_comments[["comment_id", "author_id"]].dropna()
pr_review_comment_to_author = torch.stack([
    torch.tensor(
        pr_review_comment_to_author_links["comment_id"].to_numpy(dtype=np.int64) + pr_review_comment_offset,
        dtype=torch.long,
    ),
    torch.tensor(
        pr_review_comment_to_author_links["author_id"].to_numpy(dtype=np.int64),
        dtype=torch.long,
    ),
], dim=0)

pr_review_comment_to_repo_links = past_pr_review_comments[["comment_id", "repo_id"]].dropna()
pr_review_comment_to_repo = torch.stack([
    torch.tensor(
        pr_review_comment_to_repo_links["comment_id"].to_numpy(dtype=np.int64) + pr_review_comment_offset,
        dtype=torch.long,
    ),
    torch.tensor(
        pr_review_comment_to_repo_links["repo_id"].to_numpy(dtype=np.int64) + num_users,
        dtype=torch.long,
    ),
], dim=0)

pr_review_comment_to_pr_links = past_pr_review_comments[["comment_id", "pr_id"]].dropna()
pr_review_comment_to_pr = torch.stack([
    torch.tensor(
        pr_review_comment_to_pr_links["comment_id"].to_numpy(dtype=np.int64) + pr_review_comment_offset,
        dtype=torch.long,
    ),
    torch.tensor(
        pr_review_comment_to_pr_links["pr_id"].to_numpy(dtype=np.int64) + num_users + num_repos,
        dtype=torch.long,
    ),
], dim=0)

pr_review_comment_to_review_links = past_pr_review_comments[["comment_id", "pull_request_review_id"]].dropna()
pr_review_comment_to_review = torch.stack([
    torch.tensor(
        pr_review_comment_to_review_links["comment_id"].to_numpy(dtype=np.int64) + pr_review_comment_offset,
        dtype=torch.long,
    ),
    torch.tensor(
        pr_review_comment_to_review_links["pull_request_review_id"].to_numpy(dtype=np.int64) + review_offset,
        dtype=torch.long,
    ),
], dim=0)

pr_review_comment_to_parent_links = past_pr_review_comments[["comment_id", "in_reply_to_id"]].dropna()
pr_review_comment_to_parent = torch.stack([
    torch.tensor(
        pr_review_comment_to_parent_links["comment_id"].to_numpy(dtype=np.int64) + pr_review_comment_offset,
        dtype=torch.long,
    ),
    torch.tensor(
        pr_review_comment_to_parent_links["in_reply_to_id"].to_numpy(dtype=np.int64) + pr_review_comment_offset,
        dtype=torch.long,
    ),
], dim=0)

pr_review_comment_edges = torch.cat([
    pr_review_comment_to_author, pr_review_comment_to_repo, pr_review_comment_to_pr, pr_review_comment_to_review, pr_review_comment_to_parent,
], dim=1)
reverse_pr_review_comment_edges = pr_review_comment_edges.flip(0)
pr_review_comment_edges = torch.cat([pr_review_comment_edges, reverse_pr_review_comment_edges], dim=1)

edge_index = torch.cat([edge_index, pr_review_comment_edges], dim=1)

print("Edges including reverse connections:", pr_review_comment_edges.shape)


In [ ]:
total_nodes = (
    num_users + num_repos + num_prs + num_issues
    + num_pr_reviews + num_issue_comments + num_pr_review_comments
)

# Seven node-type features plus twelve activity-count features.
x = torch.zeros((total_nodes, 19), dtype=torch.float32)

boundaries = [
    0,
    num_users,
    num_users + num_repos,
    num_users + num_repos + num_prs,
    num_users + num_repos + num_prs + num_issues,
    num_users + num_repos + num_prs + num_issues + num_pr_reviews,
    num_users + num_repos + num_prs + num_issues + num_pr_reviews + num_issue_comments,
    total_nodes,
]

for node_type in range(7):
    x[boundaries[node_type]:boundaries[node_type + 1], node_type] = 1

# Each entry tells us where an activity's user and time are stored.
activity_sources = {
    "open_pr": (past_pull_requests, "author_id", "created_at"),
    "open_issue": (
        past_issues.loc[~past_issues["is_pull_request"]],
        "author_id",
        "created_at",
    ),
    "issue_comment": (past_issue_comments, "author_id", "created_at"),
    "pr_review": (past_pr_reviews, "reviewer_id", "submitted_at"),
    "pr_review_comment": (
        past_pr_review_comments, "author_id", "created_at"
    ),
    "push": (past_pushes, "user_id", "created_at"),
}

# One summary row per user, in the same order as the user nodes.
user_activity = pd.DataFrame(index=users["user_id"])

for days in [7, 30]:
    start_date = cutoff_date - pd.Timedelta(days=days)

    for activity, (table, user_column, time_column) in activity_sources.items():
        # Keep activities that happened within this historical window.
        recent_rows = table.loc[
            (table[time_column] >= start_date) &
            (table[time_column] < cutoff_date)
        ]

        # Count how many rows belong to each user.
        counts = recent_rows[user_column].value_counts()

        # Add a named column, filling absent users with zero.
        column_name = f"{activity}_last_{days}_days"
        user_activity[column_name] = counts.reindex(
            user_activity.index, fill_value=0
        )

display(user_activity.head())

# Compress large counts, then copy the twelve columns into x.
activity_values = torch.tensor(
    user_activity.to_numpy(dtype=np.float32),
    dtype=torch.float32,
)

x[:num_users, 7:] = torch.log1p(activity_values)

print("Node features:", x.shape)
print("Graph edges:", edge_index.shape)


In [ ]:
from torch_geometric.data import Data

graph = Data(x=x, edge_index=edge_index)

print(graph)
print("Highest node number in edges:", edge_index.max().item())
print("Number of feature rows:", x.shape[0])

assert edge_index.max().item() < x.shape[0]


In [ ]:
assert graph.x.shape[1] == checkpoint["model_config"]["input_dim"]
assert edge_index.dtype == torch.long
assert edge_index.min().item() >= 0
assert edge_index.max().item() < total_nodes
assert torch.isfinite(graph.x).all()


In [ ]:
pushed_this_week = targets["user_id"].isin(week_pushes["user_id"]) # Check if each target user pushed within the week after the cutoff date
targets["push"] = pushed_this_week.astype(int) # Add a new column "push" to the targets DataFrame, where 1 indicates the user pushed and 0 indicates they did not push

pull_requests_this_week = targets["user_id"].isin(week_pull_requests["author_id"]) # Check if each target user created a pull request within the week after the cutoff date
targets["open_pr"] = pull_requests_this_week.astype(int) # Add a new column "

issue_this_week = targets["user_id"].isin(week_issues["author_id"]) # Check if each target user created an issue within the week after the cutoff date
targets["open_issue"] = issue_this_week.astype(int) # Add a new column "issue"

issue_comment_this_week = targets["user_id"].isin(week_issue_comments["author_id"]) # Check if each target user created an issue comment within the week after the cutoff date
targets["issue_comment"] = issue_comment_this_week.astype(int) # Add a new column "

pr_review_this_week = targets["user_id"].isin(week_pr_reviews["reviewer_id"]) # Check if each target user created a pull request review within the week after the cutoff date
targets["pr_review"] = pr_review_this_week.astype(int) # Add a new column "

pr_review_comment_this_week = targets["user_id"].isin(week_pr_review_comments["author_id"]) # Check if each target user created a pull request review comment within the week after the cutoff date
targets["pr_review_comment"] = pr_review_comment_this_week.astype(int) # Add a new


In [ ]:
target_nodes = torch.tensor(
    targets["user_id"].to_numpy(dtype=np.int64),
    dtype=torch.long,
)
if len(target_nodes) == 0:
    raise ValueError("No cohort users were observed before the validation cutoff.")

user_id_lookup = users.set_index("user_id")["github_user_id"]
validation_truth = targets[activity_columns].copy()
validation_truth.insert(
    0, "UserID", targets["user_id"].map(user_id_lookup).to_numpy()
)
validation_truth = validation_truth.reset_index(drop=True)
assert validation_truth["UserID"].notna().all()
assert not validation_truth["UserID"].duplicated().any()
assert validation_truth[activity_columns].isin([0, 1]).all().all()

print("Validation users:", len(validation_truth))
display(validation_truth.head())
display(validation_truth[activity_columns].sum().rename("Positive users"))


In [ ]:
graph = graph.to(device)
target_nodes = target_nodes.to(device)

model.eval()
with torch.no_grad():
    all_node_scores = model(graph)
    user_scores = all_node_scores[target_nodes]
    probabilities = torch.sigmoid(user_scores).cpu().numpy()

assert probabilities.shape == (len(validation_truth), len(activity_columns))
assert np.isfinite(probabilities).all()

validation_predictions = validation_truth[["UserID"]].copy()
validation_predictions[activity_columns] = (
    probabilities >= threshold
).astype(np.int64)

validation_probabilities = validation_truth[["UserID"]].copy()
validation_probabilities[activity_columns] = probabilities

run_date = cutoff_date.strftime("%Y-%m-%d")
predictions_path = OUTPUT_DIR / f"predictions_{run_date}.csv"
truth_path = OUTPUT_DIR / f"truth_{run_date}.csv"
probabilities_path = OUTPUT_DIR / f"probabilities_{run_date}.csv"

validation_predictions.to_csv(predictions_path, index=False)
validation_truth.to_csv(truth_path, index=False)
validation_probabilities.to_csv(probabilities_path, index=False)

display(validation_predictions.head())
display(pd.DataFrame({
    "actual_positive_users": validation_truth[activity_columns].sum(),
    "predicted_positive_users": validation_predictions[activity_columns].sum(),
}))
print("Predictions:", predictions_path)
print("Answers:", truth_path)
print("Probabilities:", probabilities_path)


In [ ]:
result = subprocess.run(
    [
        sys.executable,
        str(EVALUATOR_PATH),
        "--pred", str(predictions_path),
        "--truth", str(truth_path),
    ],
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
result.check_returncode()
(OUTPUT_DIR / f"scores_{run_date}.txt").write_text(
    result.stdout, encoding="utf-8"
)
